In [1]:
import pandas as pd

In [2]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]

In [11]:
roll_number = "1024170106"  
roll_last_two = roll_number[-2:]  
digit1, digit2 = int(roll_last_two[0]), int(roll_last_two[1])

In [6]:
categories = ["billing", "account", "general"]
cat1 = categories[digit1 % 3]
cat2 = categories[digit2 % 3]

In [7]:
personalized_entries = [
    {"question": "What is the late payment fee?", "answer": "Late payment fee is Rs 100 per month.", "keywords": "late payment fine penalty", "category": cat1},
    {"question": "Can I get a discount on annual billing?", "answer": "Yes, 10% discount if paid yearly in advance.", "keywords": "discount annual billing save", "category": cat2}
]

In [12]:
all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)

print("Final FAQ DataFrame:")
print(df)

Final FAQ DataFrame:
                                  question  \
0                   what is the annual fee   
1                    how to reset password   
2              what are your working hours   
3                    how can i pay the fee   
4            What is the late payment fee?   
5  Can I get a discount on annual billing?   

                                         answer                      keywords  \
0                     The annual fee is Rs 500.         fee cost price charge   
1              Go to Settings > Reset Password.          password reset login   
2                     We are open 9 AM to 5 PM.        hours timing open time   
3    You can pay via UPI, card, or net banking.           pay payment upi fee   
4         Late payment fee is Rs 100 per month.     late payment fine penalty   
5  Yes, 10% discount if paid yearly in advance.  discount annual billing save   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  billing  
5  billing

In [14]:
def query_score(query, df):
    query = query.lower()
    scores = []
    for idx, row in df.iterrows():
        text = (row['question'] + " " + row['answer'] + " " + row['keywords']).lower()
        words = query.split()
        score = sum(1 for w in words if w in text)
        scores.append(score)

    df_copy = df.copy()
    df_copy['score'] = scores
    return df_copy.sort_values(by='score', ascending=False)

print("Q2: Scoring results for query 'fee payment':")
result = query_score("fee payment", df)
print(result)
print("\n" + "="*50 + "\n")

Q2: Scoring results for query 'fee payment':
                                  question  \
3                    how can i pay the fee   
4            What is the late payment fee?   
0                   what is the annual fee   
1                    how to reset password   
2              what are your working hours   
5  Can I get a discount on annual billing?   

                                         answer                      keywords  \
3    You can pay via UPI, card, or net banking.           pay payment upi fee   
4         Late payment fee is Rs 100 per month.     late payment fine penalty   
0                     The annual fee is Rs 500.         fee cost price charge   
1              Go to Settings > Reset Password.          password reset login   
2                     We are open 9 AM to 5 PM.        hours timing open time   
5  Yes, 10% discount if paid yearly in advance.  discount annual billing save   

  category  score  
3  billing      2  
4  billing      2  
0  b

In [15]:
def same_category(category_name, df):
    return df[df['category'] == category_name]['question'].tolist()

print("\nQuestions in category 'billing':")
print(same_category("billing", df))


Questions in category 'billing':
['what is the annual fee', 'how can i pay the fee', 'What is the late payment fee?', 'Can I get a discount on annual billing?']


In [17]:
print("\nOriginal keywords for entry 0:", df.at[0, 'keywords'])
new_keyword = input("Enter a new keyword to add to this entry: ")
df.at[0, 'keywords'] += " " + new_keyword

roll_number = "1024170106"
df.to_csv(f"{roll_number}_faq_data.csv", index=False)
print(f"Saved to {roll_number}_faq_data.csv")


Original keywords for entry 0: fee cost price charge sale


Enter a new keyword to add to this entry:  2


Saved to 1024170106_faq_data.csv


In [18]:
print("\nFAQ count per category:")
print(df.groupby('category').size())


FAQ count per category:
category
account    1
billing    4
general    1
dtype: int64


In [19]:
def score_query_with_ties(query, df):
    query = query.lower()
    scores = []
    for idx, row in df.iterrows():
        text = (row['question'] + " " + row['answer'] + " " + row['keywords']).lower()
        words = query.split()
        score = sum(1 for w in words if w in text)
        scores.append(score)
    df_copy = df.copy()
    df_copy['score'] = scores
    max_score = df_copy['score'].max()
    if max_score == 0:
        return df_copy[df_copy['score'] == max_score]
    top_matches = df_copy[df_copy['score'] == max_score]
    return top_matches
    
print("\nTie query: 'fee payment'")
print(score_query_with_ties("fee payment", df))

print("\nNo tie query: 'reset password'")
print(score_query_with_ties("reset password", df))


Tie query: 'fee payment'
                        question                                      answer  \
3          how can i pay the fee  You can pay via UPI, card, or net banking.   
4  What is the late payment fee?       Late payment fee is Rs 100 per month.   

                    keywords category  score  
3        pay payment upi fee  billing      2  
4  late payment fine penalty  billing      2  

No tie query: 'reset password'
                question                            answer  \
1  how to reset password  Go to Settings > Reset Password.   

               keywords category  score  
1  password reset login  account      2  
